<a href="https://colab.research.google.com/github/MohamedAamer01/otto-recommender-sys/blob/main/01_Data_Understanding_OTTO_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
BASE_DIR = "/content/drive/MyDrive/OTTO_Recommender_System"

folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "results"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Structure du projet créée avec succès !")


Structure du projet créée avec succès !


In [3]:
!pip install -q --upgrade kaggle

In [4]:
from google.colab import userdata

# On récupère les secrets via leurs noms configurés dans les Secrets de Colab
try:
    username = userdata.get("KAGGLE_USERNAME")
    key = userdata.get("KAGGLE_KEY")
    print("Username configuré :", bool(username))
    print("API Key configurée :", bool(key))
except Exception as e:
    print("Erreur de configuration :", e)

Erreur de configuration : Secret KAGGLE_USERNAME does not exist.


In [5]:
!kaggle --version

Kaggle CLI 2.2.5


In [6]:
from google.colab import userdata
import os

token = userdata.get("KAGGLE_API_TOKEN")

assert token, "Token introuvable"

os.environ["KAGGLE_API_TOKEN"] = token

print("Kaggle API Token configuré avec succès !")

Kaggle API Token configuré avec succès !


In [7]:
!kaggle competitions list -v

ref,deadline,category,reward,teamCount,userHasEntered,userRank
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge,2017-12-15 23:59:00,Featured,"1,500,000 Usd",518,False,0
https://www.kaggle.com/competitions/zillow-prize-1,2018-01-10 15:59:00,Featured,"1,200,000 Usd",3770,False,0
https://www.kaggle.com/competitions/data-science-bowl-2017,2017-04-12 23:59:00,Featured,"1,000,000 Usd",1972,False,0
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection,2023-06-14 23:59:00,Featured,"1,000,000 Usd",1249,False,0
https://www.kaggle.com/competitions/gemini-3,2025-12-12 23:59:00,Featured,"500,000 Usd",4079,False,0
https://www.kaggle.com/competitions/openai-gpt-oss-20b-red-teaming,2025-08-26 23:59:00,Featured,"500,000 Usd",601,False,0
https://www.kaggle.com/competitions/openai-to-z-challenge,2025-06-29 23:59:00,Featured,"500,000 Usd",225,False,0
https://www.kaggle.com/competitions/arc-prize-2026-paper-track,2026-11-09 23:59:00,Featured,"450,000 Usd",679,False

In [8]:
#Telechargement
import os

DATA_DIR = "/content/otto_data"
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle competitions download \
    -c otto-recommender-system \
    -p /content/otto_data

100% 1.94G/1.94G [00:16<00:00, 128MB/s]



Ectraire les fichiers

In [9]:
import zipfile
import os

DATA_DIR = "/content/otto_data"
zip_path = os.path.join(DATA_DIR, "otto-recommender-system.zip")
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Extraction terminée !")
print(os.listdir(DATA_DIR))

Extraction terminée !
['otto-recommender-system.zip', 'test.jsonl', 'sample_submission.csv', 'train.jsonl']


Verifier les fichiers

In [10]:
for file in os.listdir(DATA_DIR):
    path = os.path.join(DATA_DIR, file)
    if os.path.isfile(path):
        print(f"{file} : {os.path.getsize(path) / (1024**3):.2f} Go")

otto-recommender-system.zip : 1.94 Go
test.jsonl : 0.37 Go
sample_submission.csv : 0.17 Go
train.jsonl : 10.53 Go


Examiner qlqs sessions

In [11]:
import json
train_path = os.path.join(DATA_DIR, "train.jsonl")
with open(train_path, "r") as f:
    for i in range(3):
        session = json.loads(next(f))
        print(f"\nSession ID : {session['session']}")
        print(f"Nombre d'événements : {len(session['events'])}")
        print("Premiers événements :", session["events"][:3])


Session ID : 0
Nombre d'événements : 276
Premiers événements : [{'aid': 1517085, 'ts': 1659304800025, 'type': 'clicks'}, {'aid': 1563459, 'ts': 1659304904511, 'type': 'clicks'}, {'aid': 1309446, 'ts': 1659367439426, 'type': 'clicks'}]

Session ID : 1
Nombre d'événements : 32
Premiers événements : [{'aid': 424964, 'ts': 1659304800025, 'type': 'carts'}, {'aid': 1492293, 'ts': 1659304852871, 'type': 'clicks'}, {'aid': 1492293, 'ts': 1659304863627, 'type': 'carts'}]

Session ID : 2
Nombre d'événements : 33
Premiers événements : [{'aid': 763743, 'ts': 1659304800038, 'type': 'clicks'}, {'aid': 137492, 'ts': 1659304866606, 'type': 'clicks'}, {'aid': 504789, 'ts': 1659304915207, 'type': 'clicks'}]


Pour analyser, on veut un DataFrame plat : une ligne = un événement

In [12]:
import pandas as pd
import numpy as np
from itertools import islice

rows = []
with open(train_path, "r") as f:
    for line in islice(f, 1000):
        rec = json.loads(line)
        sid = rec["session"]
        for ev in rec["events"]:
            rows.append((sid, ev["aid"], ev["ts"], ev["type"]))

df_sample = pd.DataFrame(rows, columns=["session", "aid", "ts", "type"])

# Types compacts : int32 suffit pour session et aid
df_sample["session"] = df_sample["session"].astype("int32")
df_sample["aid"] = df_sample["aid"].astype("int32")
df_sample["type"] = df_sample["type"].astype("category")

print("Shape :", df_sample.shape)
print(df_sample.dtypes)
print(f"Mémoire : {df_sample.memory_usage(deep=True).sum() / 1e6:.2f} MB")
display(df_sample.head(10))

Shape : (80267, 4)
session       int32
aid           int32
ts            int64
type       category
dtype: object
Mémoire : 1.36 MB


,session,aid,ts,type
0,0,1517085,1659304800025,clicks
1,0,1563459,1659304904511,clicks
2,0,1309446,1659367439426,clicks
3,0,16246,1659367719997,clicks
4,0,1781822,1659367871344,clicks
5,0,1152674,1659367885796,clicks
6,0,1649869,1659369893840,carts
7,0,461689,1659369898050,carts
8,0,305831,1659370027105,orders
9,0,461689,1659370027105,orders


In [13]:
#Verification
# 1. Valeurs manquantes
print(df_sample.isna().sum())

# 2. Répartition des types
print(df_sample["type"].value_counts(normalize=True))

# 3. Conversion de ts en date lisible
df_sample["datetime"] = pd.to_datetime(df_sample["ts"], unit="ms")
print(df_sample["datetime"].min(), "->", df_sample["datetime"].max())

# 4. Les événements sont-ils triés par ts dans chaque session ?
is_sorted = df_sample.groupby("session")["ts"].apply(lambda s: s.is_monotonic_increasing)
print("Sessions triées par ts :", is_sorted.mean() * 100, "%")

# 5. Taille des sessions
sizes = df_sample.groupby("session").size()
print(sizes.describe())

session    0
aid        0
ts         0
type       0
dtype: int64
type
clicks    0.896271
carts     0.081590
orders    0.022139
Name: proportion, dtype: float64
2022-07-31 22:00:00.025000 -> 2022-08-28 21:59:51.257000
Sessions triées par ts : 100.0 %
count    1000.00000
mean       80.26700
std        94.87775
min         2.00000
25%        11.00000
50%        41.00000
75%       115.00000
max       482.00000
dtype: float64


Conptage et Streaning

In [ ]:
from collections import Counter
import time

n_sessions = 0
n_events = 0
type_counts = Counter()
unsorted_sessions = 0
ts_min, ts_max = float("inf"), 0
session_sizes = []          # un entier par session : ~13M entiers, acceptable

t0 = time.time()
with open(train_path, "r") as f:
    for line in f:
        rec = json.loads(line)
        evs = rec["events"]
        n_sessions += 1
        n_events += len(evs)
        session_sizes.append(len(evs))

        ts_list = [e["ts"] for e in evs]
        if ts_list != sorted(ts_list):
            unsorted_sessions += 1
        ts_min = min(ts_min, ts_list[0])
        ts_max = max(ts_max, ts_list[-1])

        for e in evs:
            type_counts[e["type"]] += 1

        if n_sessions % 2_000_000 == 0:
            print(f"{n_sessions:,} sessions traitées ({time.time()-t0:.0f}s)")

print(f"\nSessions : {n_sessions:,}")
print(f"Événements : {n_events:,}")
print(type_counts)
print(f"Sessions non triées : {unsorted_sessions}")
print(f"Durée : {time.time()-t0:.0f}s")